# Configuration

In [42]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


# Realization

```plaintext
[ Входные токены ] ──> Shape: (Batch, SeqLen)
        │
┌───────┴─────────────────────────────────────────┐
│ ШАГ 1: Инициализация потока (Embeddings)        │
│                                                 │
│  ├─ TokenEmbedder ─────> [ hook_embed ]         │
│  └─ PositionalEmbedder ──> [ hook_pos_embed ]   │
└───────┬─────────────────────────────────────────┘
        │
        ▼
╔═════════════════════════════════════════════════╗
║   RESIDUAL STREAM (Главная магистраль данных)   ║ Shape: (Batch, SeqLen, d_model)
╚═══════╤═════════════════════════════════════════╝
        │
        │ ┌───────────────────────────────────────┐
        │ │ ШАГ 2: Transformer Block (x4 слоёв)   │
        │ │                                       │
        ├─┼─> LayerNorm                           │
        │ │      │                                │
        │ │      ▼                                │
        │ │   Attention (4 головы, bias=False)    │
        │ │      ├─ W_q, W_k, W_v ─> [ hook_q, hook_k, hook_v ]
        │ │      ├─ Q@K^T / sqrt  ─> [ hook_pattern ] (Матрица внимания)
        │ │      └─ W_o           ─> [ hook_z ]       (Выход голов до проекции)
        │ │      │                                │
        <─┼──────┘ (Сложение с потоком)           │
        │ │                                       │
        ├─┼─> LayerNorm                           │
        │ │      │                                │
        │ │      ▼                                │
        │ │   MLP (d_mlp=1024)                    │
        │ │      ├─ W_in  ─────────> [ hook_pre ]  (До активации)
        │ │      ├─ GELU                          │
        │ │      └─ W_out ─────────> [ hook_post ] (После активации)
        │ │      │                                │
        <─┼──────┘ (Сложение с потоком)           │
        │ └───────────────────────────────────────┘
        │
        ▼
┌───────┴─────────────────────────────────────────┐
│ ШАГ 3: Финальная проекция (Unembedding)         │
│                                                 │
│  ├─ Final LayerNorm                             │
│  └─ Linear (bias=False) ──> [ hook_logits ]     │
└───────┬─────────────────────────────────────────┘
        │
        ▼
[ Выходные логиты ] ──> Shape: (Batch, SeqLen, Vocab)
```

**Шаг 1. Инфраструктура хуков (Hooking System)**

- Реализация базового класса `HookPoint` (наследник `nn.Module`), который прозрачно пропускает через себя тензор, но при необходимости сохраняет его в кэш или заменяет на другой (для пэтчинга).
- Создание абстрактного базового класса `HookedRootModule(ABC, nn.Module)`, управляющего общим контекстом хуков для всей сети.

In [43]:
# Будущий путь: src/grokking_carries/interpretability/hooks.py

import contextlib
from typing import Generator

import torch
import torch.nn as nn
from torch import Tensor

# Импортируем нашу типизацию (предполагается, что пакет установлен в editable mode
# или sys.path настроен в первой ячейке ноутбука)
from grokking_carries.types import HookName, HookFunction


class HookPoint(nn.Module):
    """
    Точка перехвата активаций. Прозрачный слой, который позволяет 
    считывать или подменять проходящий через него тензор во время forward pass-а.
    """
    def __init__(self):
        super().__init__()
        self.name: HookName | None = None
        self._hooks: list[HookFunction] = []

    def set_name(self, name: HookName) -> None:
        """Устанавливает уникальный путь хука в графе модели."""
        self.name = name

    def add_hook(self, hook: HookFunction) -> None:
        """Добавляет функцию-хук в список исполнения."""
        self._hooks.append(hook)

    def remove_hook(self, hook: HookFunction) -> None:
        """Удаляет конкретный хук, если он существует."""
        if hook in self._hooks:
            self._hooks.remove(hook)

    def clear_hooks(self) -> None:
        """Очищает все хуки на данном узле."""
        self._hooks.clear()

    def forward(self, x: Tensor) -> Tensor:
        """
        Пропускает тензор через все зарегистрированные хуки.
        Если хук возвращает новый тензор, он заменяет текущий (Activation Patching).
        Если возвращает None, тензор остается неизменным (только чтение/логирование).
        """
        if not self._hooks:
            return x

        for hook in self._hooks:
            result = hook(x, self.name)
            if result is not None:
                x = result
        return x


class HookedRootModule(nn.Module):
    """
    Базовый класс для Трансформера. Автоматически индексирует все HookPoint 
    внутри себя и предоставляет безопасный интерфейс для работы с ними.
    """
    def __init__(self):
        super().__init__()
        self.mod_dict: dict[HookName, HookPoint] = {}
        self._is_setup = False

    def setup_hooks(self) -> None:
        """
        Проходит по дереву модулей, находит все HookPoint, 
        назначает им полные пути (например, 'blocks.0.attn.hook_z')
        и кэширует в словарь для быстрого доступа O(1).
        """
        self.mod_dict.clear()
        for name, module in self.named_modules():
            if isinstance(module, HookPoint):
                module.set_name(name)
                self.mod_dict[name] = module
        self._is_setup = True

    def add_hook(self, name: HookName, hook: HookFunction) -> None:
        """Глобальное добавление хука по его строковому пути."""
        if not self._is_setup:
            self.setup_hooks()
            
        if name not in self.mod_dict:
            raise KeyError(f"HookPoint с именем '{name}' не найден в модели. Доступные: {list(self.mod_dict.keys())}")
            
        self.mod_dict[name].add_hook(hook)

    def remove_all_hooks(self) -> None:
        """Полностью очищает модель от всех вмешательств."""
        for hp in self.mod_dict.values():
            hp.clear_hooks()

    @contextlib.contextmanager
    def hooks(
        self, 
        fwd_hooks: list[tuple[HookName, HookFunction]]
    ) -> Generator[None, None, None]:
        """
        Контекстный менеджер для безопасного применения хуков.
        Гарантирует, что после выхода из блока `with` добавленные хуки будут удалены,
        даже если внутри блока произошла ошибка.
        
        Пример:
        with model.hooks(fwd_hooks=[("blocks.0.attn.hook_z", my_patch_func)]):
            logits = model(tokens)
        """
        try:
            for name, hook in fwd_hooks:
                self.add_hook(name, hook)
            yield
        finally:
            # Строго удаляем только те хуки, которые добавили в этом контексте
            for name, hook in fwd_hooks:
                if name in self.mod_dict:
                    self.mod_dict[name].remove_hook(hook)

In [44]:
# Ячейка для тестирования инфраструктуры
class DummyModel(HookedRootModule):
    def __init__(self):
        super().__init__()
        self.linear = nn.Linear(10, 10)
        self.hook_mid = HookPoint() # Точка перехвата
        self.relu = nn.ReLU()
        
        self.setup_hooks() # Обязательно вызываем после инициализации всех слоев

    def forward(self, x):
        x = self.linear(x)
        x = self.hook_mid(x) # Пропускаем через хук
        return self.relu(x)

# Проверяем перехват
model = DummyModel()
x = torch.randn(1, 10)

cache = {}
def cache_hook(tensor: Tensor, name: str) -> None:
    cache[name] = tensor.detach().clone() # Сохраняем копию

with model.hooks(fwd_hooks=[("hook_mid", cache_hook)]):
    out = model(x)

assert "hook_mid" in cache
print("Успех: активации перехвачены!", cache["hook_mid"].shape)

Успех: активации перехвачены! torch.Size([1, 10])


**Шаг 2. Система эмбеддингов**

- `Token Embedder`: Базовый `nn.Embedding(vocab_size, d_model)`.
- `Positional Embedder`: Реализация позиционного кодирования.
- Тест в ноутбуке: Прогоняем батч токенов $B \times L$, убеждаемся, что на выходе получаем `Float[Tensor, "batch seq_len d_model"]`. Навешиваем `hook_embed`.

In [45]:
# Будущий путь: src/grokking_carries/model/components.py

import torch
import torch.nn as nn

# Подтягиваем наши настройки
# from grokking_carries.config import ModelConfig
# from grokking_carries.types import TokenIndices, HiddenStates
# from grokking_carries.interpretability.hooks import HookPoint

class TokenEmbedder(nn.Module):
    """Преобразует индексы токенов в векторы и кэширует результат."""
    def __init__(self, cfg: ModelConfig):
        super().__init__()
        self.embed = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.hook_embed = HookPoint()

    def forward(self, tokens: TokenIndices) -> HiddenStates:
        # tokens: [batch, seq_len]
        x = self.embed(tokens)
        return self.hook_embed(x)


class PositionalEmbedder(nn.Module):
    """Добавляет информацию об абсолютной позиции токена в последовательности."""
    def __init__(self, cfg: ModelConfig):
        super().__init__()
        self.pos_embed = nn.Embedding(cfg.max_seq_len, cfg.d_model)
        self.hook_pos_embed = HookPoint()

    def forward(self, tokens: TokenIndices) -> HiddenStates:
        batch_size, seq_len = tokens.shape
        
        # Генерируем индексы [0, 1, ..., seq_len - 1]
        # Важно: берем device у весов слоя, чтобы не упасть при переносе на GPU
        pos = torch.arange(seq_len, dtype=torch.long, device=self.pos_embed.weight.device)
        
        # Извлекаем эмбеддинги: [seq_len, d_model]
        x = self.pos_embed(pos) 
        
        # Броадкастим на размер батча: [seq_len, d_model] -> [batch, seq_len, d_model]
        # Используем expand вместо repeat для экономии памяти
        x = x.expand(batch_size, -1, -1) 
        
        return self.hook_pos_embed(x)

In [46]:
# Ячейка для тестирования эмбеддингов
cfg = ModelConfig()

# Инициализируем слои
tok_embedder = TokenEmbedder(cfg)
pos_embedder = PositionalEmbedder(cfg)

# Назначаем имена хукам (в итоговой модели это сделает HookedRootModule)
tok_embedder.hook_embed.set_name("embed")
pos_embedder.hook_pos_embed.set_name("pos_embed")

# Создаем фейковый батч: 2 примера, длина последовательности 15
dummy_tokens = torch.randint(0, cfg.vocab_size, (2, 15))

# Прогоняем forward pass
tok_out = tok_embedder(dummy_tokens)
pos_out = pos_embedder(dummy_tokens)

# Итоговый Residual Stream перед первым слоем внимания
residual_stream = tok_out + pos_out

print(f"Tokens shape: {dummy_tokens.shape} (Ожидаем: [2, 15])")
print(f"Token Embeddings shape: {tok_out.shape} (Ожидаем: [2, 15, 256])")
print(f"Pos Embeddings shape: {pos_out.shape} (Ожидаем: [2, 15, 256])")
print(f"Residual Stream shape: {residual_stream.shape} (Ожидаем: [2, 15, 256])")

# Строгая проверка типов и размерностей
assert tok_out.shape == (2, 15, cfg.d_model)
assert pos_out.shape == (2, 15, cfg.d_model)

Tokens shape: torch.Size([2, 15]) (Ожидаем: [2, 15])
Token Embeddings shape: torch.Size([2, 15, 256]) (Ожидаем: [2, 15, 256])
Pos Embeddings shape: torch.Size([2, 15, 256]) (Ожидаем: [2, 15, 256])
Residual Stream shape: torch.Size([2, 15, 256]) (Ожидаем: [2, 15, 256])


**Шаг 3. Механизм внимания (Attention)**

_Пишем с нуля, без `nn.MultiheadAttention`._

- Линейные проекции `W_q`, `W_k`, `W_v` и `W_o`.
- Функция вычисления Attention Scores: $\frac{Q K^T}{\sqrt{d\_head}}$.<sup>[[1]](https://arxiv.org/pdf/1706.03762v7)</sup>
- Применение казуальной маски, так как у нас авторегрессионная GPT-подобная модель.
- `Hook Points`: Внедряем `hook_q`, `hook_k`, `hook_v`, `hook_pattern` (матрица внимания до dropout) и `hook_z` (выход до умножения на `W_o`).

In [47]:
# Будущий путь: src/grokking_carries/model/components.py
# Добавь этот класс в тот же файл, где лежат эмбеддинги

import math
import torch
import torch.nn as nn
from torch import Tensor
import torch.nn.functional as F

# from grokking_carries.config import ModelConfig
# from grokking_carries.types import HiddenStates
# from grokking_carries.interpretability.hooks import HookPoint

class Attention(nn.Module):
    def __init__(self, cfg: ModelConfig):
        super().__init__()
        self.cfg = cfg
        
        # Линейные проекции без bias для чистоты паттернов внимания
        self.W_Q = nn.Linear(cfg.d_model, cfg.d_model, bias=False)
        self.W_K = nn.Linear(cfg.d_model, cfg.d_model, bias=False)
        self.W_V = nn.Linear(cfg.d_model, cfg.d_model, bias=False)
        self.W_O = nn.Linear(cfg.d_model, cfg.d_model, bias=False)
        
        # Точки перехвата (Hook Points)
        self.hook_q = HookPoint()
        self.hook_k = HookPoint()
        self.hook_v = HookPoint()
        self.hook_pattern = HookPoint()
        self.hook_z = HookPoint()

    def forward(self, x: HiddenStates) -> HiddenStates:
        batch, seq_len, _ = x.shape
        n_heads = self.cfg.n_heads
        d_head = self.cfg.d_head
        
        # 1. Проекция и разделение на головы: [batch, seq_len, n_heads, d_head]
        # Важно: хуки вешаются на тензоры с явно разделенными головами
        q = self.W_Q(x).view(batch, seq_len, n_heads, d_head)
        q = self.hook_q(q)
        
        k = self.W_K(x).view(batch, seq_len, n_heads, d_head)
        k = self.hook_k(k)
        
        v = self.W_V(x).view(batch, seq_len, n_heads, d_head)
        v = self.hook_v(v)
        
        # 2. Транспонирование для матричного умножения: [batch, n_heads, seq_len, d_head]
        q = q.transpose(1, 2)
        k = k.transpose(1, 2)
        v = v.transpose(1, 2)
        
        # 3. Вычисление Attention Scores: Q @ K^T / sqrt(d_head)
        # Размерность: [batch, n_heads, seq_len, seq_len]
        scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(d_head)
        
        # 4. Казуальная маска (Causal Mask)
        # device берется от входного тензора для избежания ошибок на GPU
        mask = torch.ones(seq_len, seq_len, device=x.device, dtype=torch.bool)
        mask = torch.tril(mask).view(1, 1, seq_len, seq_len)
        scores = scores.masked_fill(~mask, float('-inf'))
        
        # 5. Softmax и Hook Pattern
        pattern = F.softmax(scores, dim=-1)
        pattern = self.hook_pattern(pattern)
        
        # 6. Взвешенная сумма значений (Value)
        # Размерность: [batch, n_heads, seq_len, d_head]
        z = torch.matmul(pattern, v)
        
        # Транспонируем обратно: [batch, seq_len, n_heads, d_head]
        z = z.transpose(1, 2)
        
        # Извлекаем hook_z ДО финальной проекции, чтобы анализировать логику независимых голов
        z = self.hook_z(z)
        
        # 7. Конкатенация голов и финальная проекция
        z_flat = z.reshape(batch, seq_len, self.cfg.d_model)
        out = self.W_O(z_flat)
        
        return out

In [50]:
# Проверяем, что маска работает правильно и верхний треугольник равен 0
pattern_cache = {}
def cache_pattern(tensor: Tensor, name: str):
    pattern_cache[name] = tensor.detach().clone()

# Прямое добавление хука (так как RootModule еще не собран)
attn.hook_pattern.add_hook(cache_pattern)
attn(residual_stream) # Прогоняем forward pass
attn.hook_pattern.clear_hooks() # Обязательно убираем за собой

pattern = pattern_cache["attn.hook_pattern"]
print(f"Форма Attention Pattern: {pattern.shape} (Ожидаем: [2, 4, 15, 15])")

# Проверка казуальности (на позиции 0 токен не может смотреть на позицию 1)
assert pattern[0, 0, 0, 1].item() == 0.0, "Causal mask failed!"

print("Успех: Attention слой работает и казуальная маска корректна.")

Форма Attention Pattern: torch.Size([2, 4, 15, 15]) (Ожидаем: [2, 4, 15, 15])
Успех: Attention слой работает и казуальная маска корректна.


**Шаг 4. Полносвязный слой (MLP)**

- Два линейных слоя: расширение до `d_mlp` (1024) и сжатие обратно до `d_model` (256).
- Функция активации (GELU или ReLU).
- *Hook Points:* `hook_pre` (до активации), `hook_post` (после активации).

**Шаг 5. Сборка Transformer Block**

- Интеграция Attention и MLP.
- Добавление слоев нормализации (`LayerNorm` или `RMSNorm`). Применим до слоёв, так как это позволит рассматривать Transformer как систему параллельных вычислений в едином Residual Stream (остаточном потоке).

In [51]:
# Будущий путь: src/grokking_carries/model/components.py

class MLP(nn.Module):
    """Полносвязный слой трансформера с хуками для анализа активаций нейронов."""
    def __init__(self, cfg: ModelConfig):
        super().__init__()
        self.W_in = nn.Linear(cfg.d_model, cfg.d_mlp)
        self.W_out = nn.Linear(cfg.d_mlp, cfg.d_model)
        self.act = nn.GELU()
        
        # Точки перехвата для поиска "нейронов", отвечающих за признаки 
        # (например, нейрон, детектирующий необходимость переноса <c1>)
        self.hook_pre = HookPoint()   # До активации
        self.hook_post = HookPoint()  # После активации

    def forward(self, x: HiddenStates) -> HiddenStates:
        x = self.W_in(x)
        x = self.hook_pre(x)
        x = self.act(x)
        x = self.hook_post(x)
        return self.W_out(x)


class TransformerBlock(nn.Module):
    """Один полный блок трансформера (Pre-Norm архитектура)."""
    def __init__(self, cfg: ModelConfig):
        super().__init__()
        self.ln1 = nn.LayerNorm(cfg.d_model)
        self.attn = Attention(cfg)
        self.ln2 = nn.LayerNorm(cfg.d_model)
        self.mlp = MLP(cfg)

    def forward(self, x: HiddenStates) -> HiddenStates:
        # Residual Stream: Attention параллельно читает из потока и пишет в него
        x = x + self.attn(self.ln1(x))
        # Residual Stream: MLP читает из потока и пишет в него
        x = x + self.mlp(self.ln2(x))
        return x

In [52]:
# Ячейка для тестирования Transformer Block
block = TransformerBlock(cfg)

# Имитируем RootModule
block.mlp.hook_pre.set_name("mlp.hook_pre")
block.mlp.hook_post.set_name("mlp.hook_post")

# Прогоняем наш старый residual_stream (из Шага 2)
block_out = block(residual_stream)

print(f"Вход в блок: {residual_stream.shape}")
print(f"Выход из блока: {block_out.shape} (Ожидаем: [2, 15, 256])")

assert block_out.shape == (2, 15, cfg.d_model)
print("Успех: TransformerBlock корректно пропускает графы вычислений!")

Вход в блок: torch.Size([2, 15, 256])
Выход из блока: torch.Size([2, 15, 256]) (Ожидаем: [2, 15, 256])
Успех: TransformerBlock корректно пропускает графы вычислений!


In [53]:
# Ячейка для тестирования Transformer Block
block = TransformerBlock(cfg)

# Имитируем RootModule
block.mlp.hook_pre.set_name("mlp.hook_pre")
block.mlp.hook_post.set_name("mlp.hook_post")

# Прогоняем наш старый residual_stream (из Шага 2)
block_out = block(residual_stream)

print(f"Вход в блок: {residual_stream.shape}")
print(f"Выход из блока: {block_out.shape} (Ожидаем: [2, 15, 256])")

assert block_out.shape == (2, 15, cfg.d_model)
print("Успех: TransformerBlock корректно пропускает графы вычислений!")

Вход в блок: torch.Size([2, 15, 256])
Выход из блока: torch.Size([2, 15, 256]) (Ожидаем: [2, 15, 256])
Успех: TransformerBlock корректно пропускает графы вычислений!


**Шаг 6. Unembedding и финальная сборка**

- Сборка блоков в единый `nn.ModuleList`.
- Добавление финальной нормализации `ln_final`.
- Реализация Unembed: линейной проекции из `d_model` в `vocab_size` (19) для получения логитов. Навешиваем финальный `hook_logits`.

In [54]:
# Будущий путь: src/grokking_carries/model/transformer.py

import torch
import torch.nn as nn

from grokking_carries.config import ModelConfig
from grokking_carries.types import TokenIndices, Logits
# from grokking_carries.interpretability.hooks import HookedRootModule, HookPoint
# from grokking_carries.model.components import TokenEmbedder, PositionalEmbedder, TransformerBlock

class GrokkingCarriesTransformer(HookedRootModule):
    """
    Трансформер для задач Mechanistic Interpretability.
    Наследуется от HookedRootModule, что дает доступ к .hooks() 
    и автоматической регистрации всех точек перехвата.
    """
    def __init__(self, cfg: ModelConfig):
        super().__init__()
        self.cfg = cfg
        
        # 1. Эмбеддинги
        self.embed = TokenEmbedder(cfg)
        self.pos_embed = PositionalEmbedder(cfg)
        
        # 2. Блоки трансформера
        self.blocks = nn.ModuleList([
            TransformerBlock(cfg) for _ in range(cfg.n_layers)
        ])
        
        # 3. Финальная нормализация и Unembedding
        # В Pre-Norm архитектурах финальный LayerNorm обязателен
        self.ln_final = nn.LayerNorm(cfg.d_model)
        
        # bias=False для чистоты скалярных произведений при анализе
        self.unembed = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        
        # Хук для логитов (база для генерации и Logit Lens)
        self.hook_logits = HookPoint()
        
        # КРИТИЧЕСКИ ВАЖНО: индексируем всё дерево хуков
        self.setup_hooks()

    def forward(self, tokens: TokenIndices) -> Logits:
        # 1. Формируем начальный Residual Stream
        x = self.embed(tokens) + self.pos_embed(tokens)
        
        # 2. Пропускаем через вычислительные блоки
        for block in self.blocks:
            x = block(x)
            
        # 3. Финальная нормализация
        x = self.ln_final(x)
        
        # 4. Проекция в словарь
        logits = self.unembed(x)
        
        return self.hook_logits(logits)

**Шаг 7. Интеграционный тест (Forward Pass)**

1. **Уровень 1: Structural & Hook Smoke Test (Базовый контракт)**
* Проверка, что модель корректно инициализируется и переносится между устройствами (`cpu` <-> `cuda`).
* Автоматический подсчет и валидация всех заявленных хуков (строго сверяем количество ключей в `mod_dict` с теоретической формулой: $2 \text{ (эмбеддинги)} + N_{layers} \times 7 \text{ (внутри блоков)} + 1 \text{ (логиты)}$).


2. **Уровень 2: Numerics & Causal Integrity (Защита от утечек из будущего)**
* **NaN Check:** Проверка отсутствия `NaN` или `Inf` в логитах после Forward Pass (частая проблема при ошибках инициализации LayerNorm).
* **Causality Check (Критично):** Мы искусственно меняем токен в конце последовательности и проверяем, что логиты на предыдущих позициях остались *абсолютно идентичными*. Если это не так — сломана казуальная маска `Attention`, и модель "подглядывает" в будущее.


3. **Уровень 3: Gradient Flow (Проверка вычислительного графа)**
* Имитация шага `backward()` на фиктивном лоссе.
* Проверка того, что градиенты дотекают до самых первых слоев (например, эмбеддингов). Это гарантирует, что наши внедренные хуки не разрывают граф вычислений PyTorch.


4. **Уровень 4: Autoregressive Generation (Тест логики вывода)**
* Написание базового цикла авторегрессионной генерации (жадный поиск).
* Скармливание модели промпта `45 + 92 =` и проверка способности модели сгенерировать хотя бы `max_seq_len` токенов без падений (даже если пока это будет случайный мусор, так как сеть не обучена).

In [61]:
# Будущий путь: src/grokking_carries/tests/test_model.py (или оставить в ноутбуке как sanity check)

import torch
import torch.nn.functional as F
from grokking_carries.config import ModelConfig
# from grokking_carries.model.transformer import GrokkingCarriesTransformer

def run_fail_fast_suite():
    print("Запуск Fail-Fast интеграционных тестов...\n")
    
    cfg = ModelConfig()
    model = GrokkingCarriesTransformer(cfg).to(cfg.device)
    
    # === ТЕСТ 1: Валидация Hook Registry ===
    expected_hooks = 2 + (cfg.n_layers * 7) + 1 # embed, pos_embed + 7 на каждый слой + logits
    actual_hooks = len(model.mod_dict)
    
    assert actual_hooks == expected_hooks, \
        f"X Ошибка хуков: ожидалось {expected_hooks}, найдено {actual_hooks}."
    print(f"V Тест 1 (Hooks): Найдено {actual_hooks} точек перехвата.")

    # === ТЕСТ 2: Forward Pass & Numerics ===
    batch_size, seq_len = 2, 10
    dummy_tokens = torch.randint(0, cfg.vocab_size, (batch_size, seq_len)).to(cfg.device)
    
    logits = model(dummy_tokens)
    assert logits.shape == (batch_size, seq_len, cfg.vocab_size), \
        f"X Ошибка размерности: {logits.shape}"
    assert not torch.isnan(logits).any(), "X Обнаружены NaN в логитах!"
    assert not torch.isinf(logits).any(), "X Обнаружены Inf в логитах!"
    print("V Тест 2 (Numerics): Forward pass успешен, NaN/Inf отсутствуют.")

    # === ТЕСТ 3: Causal Integrity (Строгая проверка маски) ===
    # Копируем токены и меняем ТОЛЬКО последний токен во втором батче
    tokens_perturbed = dummy_tokens.clone()
    tokens_perturbed[1, -1] = (tokens_perturbed[1, -1] + 1) % cfg.vocab_size
    
    logits_perturbed = model(tokens_perturbed)
    
    # Проверяем, что логиты ВСЕХ предыдущих позиций остались идентичными
    diff = (logits[1, :-1] - logits_perturbed[1, :-1]).abs().max()
    assert diff < 1e-6, \
        f"X Causal Mask сломана! Изменение токена в будущем повлияло на прошлое (diff={diff})."
    print("V Тест 3 (Causality): Казуальная маска работает безупречно.")

    # === ТЕСТ 4: Gradient Flow (Проверка графа вычислений) ===
    loss = logits.sum() # Фиктивный лосс
    loss.backward()
    
    assert model.embed.embed.weight.grad is not None, "X Градиенты не доходят до TokenEmbedder!"
    assert model.unembed.weight.grad is not None, "X Градиенты не доходят до Unembed!"
    
    # Очищаем градиенты после теста
    model.zero_grad()
    print("V Тест 4 (Gradients): Вычислительный граф цел, backprop работает.")

    # === ТЕСТ 5: Autoregressive Generation ===
    print("\nПроверка цикла генерации...")
    
    # Имитируем промпт "45 + 92 ="
    # Допустим индексы: 0=pad, 4=4, 5=5, 10=+, 9=9, 2=2, 11==
    prompt_tokens = torch.tensor([[4, 5, 10, 9, 2, 11]], dtype=torch.long).to(cfg.device)
    generated = prompt_tokens.clone()
    
    max_new_tokens = 5
    model.eval() # Отключаем dropout (если бы он был)
    with torch.no_grad():
        for _ in range(max_new_tokens):
            out_logits = model(generated)
            # Берем логит последнего токена
            next_token = torch.argmax(out_logits[:, -1, :], dim=-1, keepdim=True)
            generated = torch.cat([generated, next_token], dim=1)
            
    assert generated.shape == (1, prompt_tokens.shape[1] + max_new_tokens), \
        "X Ошибка в цикле генерации авторегрессии!"
    
    print(f"Промпт: {prompt_tokens[0].tolist()}")
    print(f"Сгенерировано (включая промпт): {generated[0].tolist()}")
    print("V Тест 5 (Generation): Авторегрессионный цикл работает штатно.")
    
    print("\nУСПЕХ: Все подсистемы Трансформера функционируют корректно.")

# Запуск
run_fail_fast_suite()

Запуск Fail-Fast интеграционных тестов...

V Тест 1 (Hooks): Найдено 31 точек перехвата.
V Тест 2 (Numerics): Forward pass успешен, NaN/Inf отсутствуют.
V Тест 3 (Causality): Казуальная маска работает безупречно.
V Тест 4 (Gradients): Вычислительный граф цел, backprop работает.

Проверка цикла генерации...
Промпт: [4, 5, 10, 9, 2, 11]
Сгенерировано (включая промпт): [4, 5, 10, 9, 2, 11, 18, 7, 18, 5, 13]
V Тест 5 (Generation): Авторегрессионный цикл работает штатно.

УСПЕХ: Все подсистемы Трансформера функционируют корректно.
